# Unit 3 – Practice & Consolidation Worksheet

These exercises are for your own practice and are **not graded**. Work through them after completing the Unit 3 reading material. Attempt each exercise before revealing the answer.

## Exercise types

| Type | What to do |
|---|---|
| Predict the output | Write what you think the code will print *before* running the cell |
| Fix the bug | The code contains an error — find and correct it |
| Fill in the blank | Complete the missing line(s) to produce the expected output |
| Adapt and extend | A working example is given — modify it to solve a related problem |

**Topics covered:**
1. DataFrame creation and inspection
2. Selecting and filtering
3. Adding, modifying, and dropping columns and rows
4. Reshaping, joining, and groupby
5. Data cleaning: duplicates, missing values, and outliers

---

## Shared Dataset

All exercises in this notebook use the same retail orders dataset. Run the cell below **first** — you will need it for every section.

In [1]:
import pandas as pd
import numpy as np

np.random.seed(0)

data = {
    'order_id':    list(range(1001, 1036)),
    'customer':    ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve',
                    'Frank', 'Grace', 'Henry', 'Isla', 'James',
                    'Karen', 'Leo', 'Mia', 'Noah', 'Olivia',
                    'Alice', 'Bob', 'Charlie', 'Diana', 'Eve',
                    'Frank', 'Grace', 'Henry', 'Isla', 'James',
                    'Karen', 'Leo', 'Mia', 'Noah', 'Olivia',
                    'Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'category':    (['Electronics', 'Books', 'Clothing', 'Books', 'Electronics'] * 7),
    'quantity':    [2, 1, 3, 1, 2, 4, 1, 2, 3, 1,
                    2, 5, 1, 2, 3, 1, 2, 1, 4, 2,
                    3, 1, 2, 1, 3, 2, 1, 4, 2, 1,
                    3, 2, 1, 2, 3],
    'unit_price':  [299.99, 14.99, 49.99, 9.99, 199.99,
                    599.99, 29.99, 149.99, 79.99, 12.99,
                    349.99, 24.99, 59.99, 8.99, 129.99,
                    None,   14.99, 49.99, 9.99, None,
                    599.99, 29.99, None,  79.99, 12.99,
                    349.99, 24.99, 59.99, None,  129.99,
                    299.99, 14.99, 49.99, 9.99, 199.99],
    'region':      (['North', 'South', 'East', 'West', 'North'] * 7),
    'returned':    ([False, False, True, False, False,
                     False, True,  False, False, True] * 3 + [False, False, False, False, False]),
}

# Introduce a few duplicates
df_base = pd.DataFrame(data)
dup_rows = df_base.iloc[[0, 7]].copy()
df = pd.concat([df_base, dup_rows], ignore_index=True)

# Introduce one extreme outlier
df.loc[2, 'unit_price'] = 9999.00

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (37, 7)


,order_id,customer,category,quantity,unit_price,region,returned
0,1001,Alice,Electronics,2,299.99,North,False
1,1002,Bob,Books,1,14.99,South,False
2,1003,Charlie,Clothing,3,9999.00,East,True
3,1004,Diana,Books,1,9.99,West,False
4,1005,Eve,Electronics,2,199.99,North,False


---
## Section 1 – DataFrame Creation and Inspection

### Exercise 1.1 Predict the output

Without running the cell, write down what you expect this code to print. Then run it to check.

```python
print(df.shape)
print(type(df['category']))
```

<details>
<summary><strong>Select here for answer</strong></summary>

```python
print(df.shape)      # (37, 7)
print(type(df['category']))  # <class 'pandas.core.series.Series'>
```

`df.shape` returns a tuple `(rows, columns)`. The base dataset has 35 rows, plus 2 duplicate rows added in setup, giving 37. Selecting a single column with `df['column_name']` always returns a `Series`, not a `DataFrame`.

</details>

### Exercise 1.2 Fill in the blank

Complete the code below so that it:
- shows the **last** 4 rows of the DataFrame, and
- prints the **data type** of each column.

```python
print(df.tail(4))
df.info()
```

<details>
<summary><strong>Select here for answer</strong></summary>

```python
print(df.tail(4))
df.info()
```

`tail(n)` returns the last `n` rows. `info()` prints column names, non-null counts, and data types, useful as a first check on any new dataset.

</details>

### Exercise 1.3 Fix the bug

The code below is meant to show summary statistics for **only** the numerical columns. It raises a `TypeError`. Find and fix the error.

```python
df.describe(include='number')
```

<details>
<summary><strong>Select here for answer</strong></summary>

```python
df.describe(include='number')
```

The correct value is `'number'` (singular), not `'numbers'`. Alternatively, omit the parameter entirely — `describe()` defaults to numerical columns only.

</details>

### Exercise 1.4 Adapt and extend

The example below creates a small DataFrame from a dictionary of lists.

```python
sample = pd.DataFrame({
    'product': ['Laptop', 'Phone', 'Tablet'],
    'price':   [999, 599, 349]
})
```

Adapt it to create a DataFrame called `returns_log` with three columns — `order_id`, `reason`, and `refund_amount` — containing at least four rows of plausible data. Then use `info()` to confirm the column types.

<details>
<summary><strong>Select here for answer</strong></summary>

```python
returns_log = pd.DataFrame({
    'order_id':      [1001, 1007, 1010, 1021],
    'reason':        ['Damaged', 'Wrong item', 'Changed mind', 'Faulty'],
    'refund_amount': [299.99, 29.99, 79.99, 149.99]
})
returns_log.info()
```

Notice that `order_id` and `refund_amount` are numeric, while `reason` is `object` (Pandas' label for string columns). The column types match what we would expect from the values provided.

</details>

### Exercise 1.5 Predict the output

Predict the output of the following, then run the cell to verify.

```python
print(df['unit_price'].describe())
```

Pay attention to the `count` value. Why is it less than 37?

<details>
<summary><strong>Select here for answer</strong></summary>

```python
# count will be 33, not 37, because describe() excludes NaN values from all statistics
```

`describe()` silently ignores `NaN` values. The `count` row shows how many **non-null** values exist, so any column with missing data will show a count lower than the total number of rows. This is a useful way to spot missing values quickly.

</details>

---
## Section 2 – Selecting and Filtering

### Exercise 2.1 Fix the bug

The code is meant to select only the `customer` and `unit_price` columns. It raises a `KeyError`. Fix it.

```python
df['customer', 'unit_price']
```

<details>
<summary><strong>Select here for answer</strong></summary>

```python
df[['customer', 'unit_price']]
```

Selecting **multiple** columns requires a **list** inside the square brackets: `df[['col1', 'col2']]`. Using `df['col1', 'col2']` (without the inner list) passes a tuple as the key, which Pandas does not recognise as a column selector.

</details>

### Exercise 2.2 Fill in the blank

Complete the filter to return all orders where the `category` is `'Electronics'` **and** the `quantity` is greater than 1.

```python
electronics_bulk = df[(df[________] == ________) & (________)]
print(electronics_bulk.shape)
```

<details>
<summary><strong>Select here for answer</strong></summary>

```python
electronics_bulk = df[(df['category'] == 'Electronics') & (df['quantity'] > 1)]
print(electronics_bulk.shape)
```

When combining two conditions, each must be wrapped in parentheses and joined with `&` (not the Python keyword `and`). Using `and` on two Series raises a `ValueError`.

</details>

### Exercise 2.3 Predict the output

Predict what the following two expressions return, then run them to check. They look similar — are the results the same type?

```python
result_a = df.iloc[0]
result_b = df.iloc[[0]]

print(type(result_a))
print(type(result_b))
```

<details>
<summary><strong>Select here for answer</strong></summary>

```python
# result_a → <class 'pandas.core.series.Series'>
# result_b → <class 'pandas.core.frame.DataFrame'>
```

`iloc[0]` with a single integer returns a **Series** (one row as a labelled sequence). `iloc[[0]]` with a list returns a **DataFrame** (a table with one row). This distinction matters when passing results to functions that expect a specific type.

</details>

### Exercise 2.4 Fill in the blank

Use `isin()` to select only orders from the `'North'` and `'East'` regions, then display only the `order_id`, `region`, and `unit_price` columns.

```python
north_east = df[df[________].isin(________)]
print(north_east[[________]])
```

<details>
<summary><strong>Select here for answer</strong></summary>

```python
north_east = df[df['region'].isin(['North', 'East'])]
print(north_east[['order_id', 'region', 'unit_price']])
```

`isin()` is cleaner than writing `(df['region'] == 'North') | (df['region'] == 'East')`, and scales well when checking membership against many values.

</details>

### Exercise 2.5 Adapt and extend

The example below selects orders where `returned` is `True`.

```python
returned_orders = df[df['returned'] == True]
```

Adapt it to create a DataFrame called `high_value` containing only orders where `unit_price` is above 100 **and** the order has **not** been returned. Then print how many rows it contains.

<details>
<summary><strong>Select here for answer</strong></summary>

```python
high_value = df[(df['unit_price'] > 100) & (df['returned'] == False)]
print(len(high_value))
```

Note that `unit_price` has some `NaN` values, so rows with missing prices are automatically excluded by the `> 100` comparison. You could also write `~df['returned']` as a more idiomatic alternative to `df['returned'] == False`.

</details>

---
## Section 3 – Adding, Modifying, and Dropping Columns and Rows

### Exercise 3.1 Fill in the blank

Add a new column called `total_price` to `df`, calculated as `quantity * unit_price`. Then display the first three rows showing only `order_id`, `quantity`, `unit_price`, and `total_price`.

```python
df[________] = ________
df[['order_id', 'quantity', 'unit_price', 'total_price']].________
```

<details>
<summary><strong>Select here for answer</strong></summary>

```python
df['total_price'] = df['quantity'] * df['unit_price']
df[['order_id', 'quantity', 'unit_price', 'total_price']].head(3)
```

Pandas performs the multiplication **element-wise** across the whole column without a loop. Rows where `unit_price` is `NaN` will produce `NaN` in `total_price` as well.

</details>

### Exercise 3.2 Fix the bug

The code below tries to rename the column `returned` to `is_returned`, but raises an error. Fix it.

```python
df.rename('returned', 'is_returned', inplace=True)
```

<details>
<summary><strong>Select here for answer</strong></summary>

```python
df.rename(columns={'returned': 'is_returned'}, inplace=True)
```

`rename()` takes a **dictionary** passed to the `columns` parameter, mapping old names to new names. Passing two positional string arguments has no matching signature and raises a `TypeError`.

</details>

### Exercise 3.3 Predict the output

The code below drops two columns. Without running it, predict which columns remain in `df_trimmed`, and confirm by running the cell.

```python
df_trimmed = df.drop(columns=['region', 'is_returned'])
print(df_trimmed.columns.tolist())
```

*(Note: rename from Exercise 3.2 must have been run first for `is_returned` to exist.)*

<details>
<summary><strong>Select here for answer</strong></summary>

```python
# ['order_id', 'customer', 'category', 'quantity', 'unit_price', 'total_price']
# (assuming total_price was added in Exercise 3.1)
```

`drop()` returns a **new** DataFrame without modifying the original (unless `inplace=True`). All remaining columns appear in their original order.

</details>

### Exercise 3.4 Adapt and extend

The example below updates a single cell using `at[]`:

```python
df.at[0, 'quantity'] = 10
```

Adapt it to do the following:
1. Update the `unit_price` for row index `5` to `499.99`.
2. Add a new column `discount` with the value `0.0` for every row.
3. Then set the `discount` to `0.1` for any row where `unit_price` (original, not updated) is above `200`.

<details>
<summary><strong>Select here for answer</strong></summary>

```python
df.at[5, 'unit_price'] = 499.99
df['discount'] = 0.0
df.loc[df['unit_price'] > 200, 'discount'] = 0.1
```

`at[]` is the fastest way to update a single cell by index label and column name. `loc[]` with a boolean condition on the left-hand side of assignment is the standard way to update a whole subset of rows at once.

</details>

### Exercise 3.5 Fill in the blank

Use `concat()` to append the row below as a new order at the bottom of `df`. Set `ignore_index=True` so that row numbering continues automatically.

```python
new_order = pd.Series({
    'order_id': 1999, 'customer': 'Zara', 'category': 'Books',
    'quantity': 2, 'unit_price': 19.99, 'region': 'South',
    'is_returned': False, 'total_price': 39.98, 'discount': 0.0
})

df = pd.concat([df, ________], ignore_index=________)
print(df.tail(2))
```

<details>
<summary><strong>Select here for answer</strong></summary>

```python
df = pd.concat([df, new_order.to_frame().T], ignore_index=True)
print(df.tail(2))
```

`concat()` expects objects with compatible orientations. A `Series` is column-shaped by default, so `.to_frame().T` transposes it into a single-row DataFrame before concatenation. `ignore_index=True` resets the row numbers so the new row gets the next sequential index.

If you find `.to_frame().T` unintuitive, a cleaner alternative in modern Pandas is to pass a list of dictionaries directly:

```python
df = pd.concat([df, pd.DataFrame([new_order])], ignore_index=True)
```

This avoids the transpose step entirely and reads more naturally when adding one or more rows.

</details>

---
## Section 4 – Reshaping, Joining, and GroupBy

### Exercise 4.1 Predict the output

Predict what the following `groupby` returns — specifically, how many rows the result will have and what the index will be.

```python
result = df.groupby('category')['unit_price'].mean()
print(result)
```

<details>
<summary><strong>Select here for answer</strong></summary>

```python
# 3 rows — one for each unique category: Books, Clothing, and Electronics.
# The index is the category name
# NaN values in unit_price are ignored by default
```

`groupby().mean()` produces one row per unique group value. The group column becomes the index. Missing values are excluded from the mean calculation automatically.

</details>

### Exercise 4.2 Fill in the blank

Calculate the **total** `quantity` sold per `region`, sorted from highest to lowest.

```python
region_totals = df.groupby(________)[________].________().sort_values(________=False)
print(region_totals)
```

<details>
<summary><strong>Select here for answer</strong></summary>

```python
region_totals = df.groupby('region')['quantity'].sum().sort_values(ascending=False)
print(region_totals)
```

Chaining `.sort_values(ascending=False)` directly after the aggregation is clean and idiomatic. By default `sort_values` on a Series sorts by value; `ascending=False` gives descending order.

</details>

### Exercise 4.3 Fix the bug

The code below is meant to do an inner join between `df` and the `discounts` table on `category`. It raises a `MergeError`. Fix it.

```python
discounts = pd.DataFrame({
    'cat':        ['Books', 'Clothing', 'Electronics'],
    'category_discount': [0.05, 0.10, 0.15]
})

merged = pd.merge(df, discounts)
```

<details>
<summary><strong>Select here for answer</strong></summary>

```python
merged = pd.merge(df, discounts, left_on='category', right_on='cat')
```

The two DataFrames do not share a column name, so `pandas` cannot infer the join key and raises a `MergeError`. You must specify which column to use from each DataFrame using `left_on` and `right_on`. `on='column_name'` can only be used when both DataFrames share the same column name for the join key.

</details>

### Exercise 4.4 Adapt and extend

The example below uses `melt()` to reshape a wide DataFrame into a long one.

```python
wide = pd.DataFrame({
    'product': ['Laptop', 'Phone'],
    'Q1': [100, 200],
    'Q2': [150, 180]
})
long = wide.melt(id_vars='product', var_name='quarter', value_name='sales')
```

Apply the same idea to a small summary table you create from `df`: first use `groupby` and `unstack()` to produce a wide table showing **total quantity** by `category` (rows) and `region` (columns), then melt it back to long format.

<details>
<summary><strong>Select here for answer</strong></summary>

```python
wide_summary = (df.groupby(['category', 'region'])['quantity']
                  .sum()
                  .unstack(fill_value=0))

long_summary = wide_summary.reset_index().melt(
    id_vars='category', var_name='region', value_name='total_quantity'
)
print(long_summary.head(10))
```

`unstack()` pivots the innermost index level into columns, turning the grouped result into a wide table. `melt()` then reverses that step. Being comfortable with both directions is essential when preparing data for visualisation or machine learning.

</details>

### Exercise 4.5 Predict the output

Predict the **shape** of the result before running the cell.

```python
df2 = pd.DataFrame({
    'order_id': [1001, 1004, 9999],
    'delivery_days': [3, 5, 2]
})

result = pd.merge(df, df2, on='order_id', how='left')
print(result.shape)
```

<details>
<summary><strong>Select here for answer</strong></summary>

```python
# result.shape → (37, <original_cols + 1>)
# e.g. (37, 10) if df currently has 9 columns
# Row 9999 from df2 is dropped because it has no match in df
```

A **left join** keeps all rows from the left DataFrame (`df`) and matches them with rows from the right DataFrame (`df2`) where `order_id` matches. Rows in `df2` with no match in `df` (order 9999) are discarded. Rows in `df` with no match in `df2` get `NaN` in the `delivery_days` column.

</details>

---
## Section 5 – Data Cleaning: Duplicates, Missing Values, and Outliers

### Exercise 5.1 Fill in the blank

The shared dataset was deliberately built with 2 duplicate rows. Write the code to:
1. Print how many duplicate rows exist.
2. Remove them, keeping the first occurrence.
3. Confirm the new shape.

```python
print(df.duplicated().________)
df = df.drop_duplicates(keep=________)
print(df.________)
```

<details>
<summary><strong>Select here for answer</strong></summary>

```python
print(df.duplicated().sum())
df = df.drop_duplicates(keep='first')
print(df.shape)
```

`duplicated()` returns a boolean Series (True for every row that is a duplicate of an earlier row). `.sum()` counts the `True` values. `drop_duplicates(keep='first')` retains the first occurrence of each duplicated row and removes the rest.

</details>

### Exercise 5.2 Predict the output

Before running the cell, predict what the following prints — specifically, which columns have missing values and approximately how many.

```python
print(df.isnull().sum())
```

<details>
<summary><strong>Select here for answer</strong></summary>

```python
# unit_price: 4 missing values (introduced in setup)
# All other columns: 0
# (total_price will also show 4 if it was derived from unit_price)
```

`isnull()` returns a boolean DataFrame (True where the value is NaN). Calling `.sum()` on it adds up the True values column by column, giving a count of missing values per column.

</details>

### Exercise 5.3 Fix the bug

The code below fills missing `unit_price` values with the column mean, but it does not update `df`. Fix it so the missing values are actually replaced.

```python
df['unit_price'].fillna(df['unit_price'].mean())
print(df['unit_price'].isnull().sum())  # still shows 4
```

<details>
<summary><strong>Select here for answer</strong></summary>

```python
df['unit_price'] = df['unit_price'].fillna(df['unit_price'].mean())
print(df['unit_price'].isnull().sum())  # now shows 0
```

`fillna()` returns a **new Series** — it does not modify the original in place. You must assign the result back to the column. (You could alternatively pass `inplace=True`, but reassigning is generally clearer.)

</details>

### Exercise 5.4 Adapt and extend

The IQR method for detecting outliers works as follows:

```python
Q1 = df['unit_price'].quantile(0.25)
Q3 = df['unit_price'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = df[(df['unit_price'] < lower) | (df['unit_price'] > upper)]
print(outliers[['order_id', 'unit_price']])
```

Run this and confirm it detects the outlier introduced in setup (row 2, price £9,999). Then adapt it to:
1. Print the **percentage** of rows that are outliers.
2. Create a new DataFrame `df_clean` with the outliers removed.

<details>
<summary><strong>Select here for answer</strong></summary>

```python
Q1 = df['unit_price'].quantile(0.25)
Q3 = df['unit_price'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outlier_mask = (df['unit_price'] < lower) | (df['unit_price'] > upper)
print(f"Outlier rows: {outlier_mask.sum()} ({outlier_mask.sum() / len(df) * 100:.1f}%)")

df_clean = df[~outlier_mask].copy()
print(f"Rows after removing outliers: {len(df_clean)}")
```

`~outlier_mask` inverts the boolean Series, selecting all rows that are **not** outliers. Using `.copy()` creates an independent DataFrame so that later modifications to `df_clean` do not affect `df`.

</details>

### Exercise 5.5 Fill in the blank

Complete the code below to fill missing `unit_price` values using the **median price for the same category** (group-based imputation), rather than the global mean.

```python
df['unit_price'] = df.groupby(________)[________].transform(
    lambda x: x.fillna(________)
)
print(df['unit_price'].isnull().sum())
```

<details>
<summary><strong>Select here for answer</strong></summary>

```python
df['unit_price'] = df.groupby('category')['unit_price'].transform(
    lambda x: x.fillna(x.median())
)
print(df['unit_price'].isnull().sum())
```

`transform()` applies a function to each group and returns a Series aligned with the original index — unlike `agg()`, which collapses the group into a single value. Using the group median is more accurate than the global median when categories have different typical prices (e.g., Electronics vs Books).

</details>


## Well done for completing Unit 3 exercises!

If any section felt difficult, revisit the relevant part of the Unit 3 reading material before moving on.

**Key reminders:**

- Selecting a single column with `df['col']` returns a **Series**; selecting multiple columns with `df[['col1', 'col2']]` returns a **DataFrame** — the extra pair of brackets matters
- When combining boolean filters, wrap each condition in parentheses and use `&` or `|`, not `and` or `or`
- `iloc[0]` returns a **Series** (one row); `iloc[[0]]` returns a **DataFrame** (a table with one row) — the difference matters when passing results to other functions
- `drop()`, `rename()`, and `fillna()` return a **new** object by default and do not modify the original; you must either reassign the result or pass `inplace=True`
- `concat()` appends a Series as a row only after calling `.to_frame().T` to give it the correct orientation
- `groupby().agg()` collapses groups to single values; `groupby().transform()` returns a Series aligned with the original index — use `transform()` when you need to fill or scale values in place
- In a `merge()`, always specify `on='column_name'` explicitly — relying on Pandas to infer the join key can cause unexpected results or errors
- `duplicated().sum()` counts duplicate rows; `drop_duplicates(keep='first')` removes them while retaining the first occurrence
- `isnull().sum()` gives a per-column count of missing values — always run this before any analysis or modelling
- When using the IQR method, rows where the value is `NaN` are excluded from the outlier check automatically — handle missing values before or after outlier removal, not together